In [1]:
# ============================================================
# PRE-ABLATION SETUP CELL
# Creates:
#   train_loader, val_loader, test_loader
#   CSI_CHANNELS, NUM_CLASSES
#   LABEL_MAPPING, INV_LABEL_MAPPING
#
# Run this BEFORE the DRFT-LSTM ablation cell.
# ============================================================

import os
import re
import gc
import glob
import json
import h5py
import random
import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader

# ============================================================
# CONFIG
# ============================================================

TASK_NAME = "HumanActivityRecognition"

SEED = 42

BATCH_SIZE = 8
EVAL_BATCH_SIZE = 32
NUM_WORKERS = 0

TARGET_SUBCARRIERS = 56
TARGET_TIME_LEN = 500

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("medium")

# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


set_seed(SEED)
cleanup()

# ============================================================
# AUTO-DETECT CSI-BENCH ROOT
# ============================================================

def find_task_root(task_name="HumanActivityRecognition"):
    patterns = [
        f"/kaggle/input/datasets/guozhenjennzhu/csi-bench/Multitask/{task_name}",
        f"/kaggle/input/**/Multitask/{task_name}",
        f"/kaggle/input/**/{task_name}",
    ]

    candidates = []

    for pattern in patterns:
        for path in glob.glob(pattern, recursive=True):
            if os.path.isdir(path):
                candidates.append(path)

    candidates = list(dict.fromkeys(candidates))

    print("\nCandidate task roots:")
    for path in candidates:
        print(" -", path)

    for path in candidates:
        split_file = os.path.join(path, "splits", "train_id.json")
        metadata_file = os.path.join(path, "metadata", "sample_metadata.csv")

        if os.path.exists(split_file) and os.path.exists(metadata_file):
            print("\n✅ Using ROOT:")
            print(path)
            return path

    raise FileNotFoundError(
        "Could not find HumanActivityRecognition root with splits/train_id.json "
        "and metadata/sample_metadata.csv"
    )


ROOT = find_task_root(TASK_NAME)

METADATA_DIR = os.path.join(ROOT, "metadata")
SPLITS_DIR = os.path.join(ROOT, "splits")
MULTITASK_DIR = os.path.dirname(ROOT)
CSI_BENCH_DIR = os.path.dirname(MULTITASK_DIR)

print("\n========== ROOT VERIFICATION ==========")
print("ROOT:", ROOT)
print("Has metadata:", os.path.exists(METADATA_DIR))
print("Has splits:", os.path.exists(SPLITS_DIR))
print("Train split:", os.path.exists(os.path.join(SPLITS_DIR, "train_id.json")))
print("Val split:", os.path.exists(os.path.join(SPLITS_DIR, "val_id.json")))
print("Test split:", os.path.exists(os.path.join(SPLITS_DIR, "test_id.json")))
print("Metadata CSV:", os.path.exists(os.path.join(METADATA_DIR, "sample_metadata.csv")))
print("sub_Human_h5:", os.path.exists(os.path.join(MULTITASK_DIR, "sub_Human_h5")))

# ============================================================
# SPLIT + LABEL SETUP
# ============================================================

def load_split_ids(root_dir, split):
    split_file = os.path.join(root_dir, "splits", f"{split}_id.json")

    if not os.path.exists(split_file):
        raise FileNotFoundError(f"Missing split file: {split_file}")

    with open(split_file, "r") as f:
        return set(map(str, json.load(f)))


metadata_path = os.path.join(ROOT, "metadata", "sample_metadata.csv")
metadata = pd.read_csv(metadata_path)

metadata["id"] = metadata["id"].astype(str)
metadata["file_path"] = metadata["file_path"].astype(str)

train_ids = load_split_ids(ROOT, "train")
val_ids = load_split_ids(ROOT, "val")
test_ids = load_split_ids(ROOT, "test")

print("\n========== SPLIT CHECK ==========")
print("Train IDs:", len(train_ids))
print("Val IDs  :", len(val_ids))
print("Test IDs :", len(test_ids))
print("Train-Val overlap :", len(train_ids & val_ids))
print("Train-Test overlap:", len(train_ids & test_ids))
print("Val-Test overlap  :", len(val_ids & test_ids))

train_meta = metadata[metadata["id"].isin(train_ids)].copy()
train_labels = sorted(train_meta["label"].unique())

LABEL_MAPPING = {label: idx for idx, label in enumerate(train_labels)}
INV_LABEL_MAPPING = {v: k for k, v in LABEL_MAPPING.items()}
NUM_CLASSES = len(LABEL_MAPPING)

print("\nLabel mapping:")
print(LABEL_MAPPING)

# ============================================================
# DATASET
# ============================================================

class CSIBenchDataset(Dataset):
    def __init__(
        self,
        root_dir,
        split="train",
        normalize=True,
        target_subcarriers=56,
        target_time_len=500,
        label_mapping=None,
    ):
        self.root_dir = root_dir
        self.split = split
        self.normalize = normalize
        self.target_subcarriers = target_subcarriers
        self.target_time_len = target_time_len
        self.label_mapping = label_mapping

        self.metadata_dir = os.path.join(root_dir, "metadata")
        self.splits_dir = os.path.join(root_dir, "splits")
        self.multitask_dir = os.path.dirname(root_dir)
        self.csi_bench_dir = os.path.dirname(self.multitask_dir)

        split_file = os.path.join(self.splits_dir, f"{split}_id.json")
        metadata_file = os.path.join(self.metadata_dir, "sample_metadata.csv")

        if not os.path.exists(split_file):
            raise FileNotFoundError(f"Split file not found: {split_file}")

        if not os.path.exists(metadata_file):
            raise FileNotFoundError(f"Metadata file not found: {metadata_file}")

        with open(split_file, "r") as f:
            self.sample_ids = list(map(str, json.load(f)))

        self.metadata = pd.read_csv(metadata_file)
        self.metadata["id"] = self.metadata["id"].astype(str)
        self.metadata["file_path"] = self.metadata["file_path"].astype(str)

        self.meta_dict = {
            str(row["id"]): row
            for _, row in self.metadata.iterrows()
        }

        print(f"Loaded {len(self.sample_ids)} samples for split: {split}")

    def __len__(self):
        return len(self.sample_ids)

    def resolve_file_path(self, raw_path):
        raw = str(raw_path).replace("\\", "/").strip()

        # Important fix for paths like:
        # ../../sub_Human_h5/user_U01/...
        while raw.startswith("../"):
            raw = raw[3:]

        if raw.startswith("./"):
            raw = raw[2:]

        candidates = []

        if os.path.isabs(raw):
            candidates.append(os.path.normpath(raw))

        candidates.extend([
            os.path.normpath(os.path.join(self.metadata_dir, raw)),
            os.path.normpath(os.path.join(self.root_dir, raw)),
            os.path.normpath(os.path.join(self.multitask_dir, raw)),
            os.path.normpath(os.path.join(self.csi_bench_dir, raw)),
            os.path.normpath(os.path.join("/kaggle/input", raw)),
        ])

        if "sub_Human_h5/" in raw:
            suffix = raw.split("sub_Human_h5/", 1)[1]
            candidates.append(
                os.path.normpath(
                    os.path.join(self.multitask_dir, "sub_Human_h5", suffix)
                )
            )
            candidates.append(
                os.path.normpath(
                    os.path.join(self.csi_bench_dir, "Multitask", "sub_Human_h5", suffix)
                )
            )

        if "sub_Human_mat/" in raw:
            suffix = raw.split("sub_Human_mat/", 1)[1]
            candidates.append(
                os.path.normpath(
                    os.path.join(self.multitask_dir, "sub_Human_mat", suffix)
                )
            )
            candidates.append(
                os.path.normpath(
                    os.path.join(self.csi_bench_dir, "Multitask", "sub_Human_mat", suffix)
                )
            )

        for path in candidates:
            if os.path.exists(path):
                return path

        base = os.path.basename(raw)
        matches = []

        for search_base in [self.multitask_dir, self.csi_bench_dir, "/kaggle/input"]:
            pattern = os.path.join(search_base, "**", base)
            matches.extend(glob.glob(pattern, recursive=True))

        matches = sorted(list(set(matches)))

        if len(matches) == 1:
            return matches[0]

        if len(matches) > 1:
            raise RuntimeError(
                "Ambiguous file resolution. Multiple files share the same basename.\n"
                f"metadata file_path: {raw_path}\n"
                f"cleaned path: {raw}\n"
                f"basename: {base}\n"
                "Matches:\n" + "\n".join(matches[:20])
            )

        raise FileNotFoundError(
            "Could not resolve CSI file path.\n"
            f"metadata file_path: {raw_path}\n"
            f"cleaned path: {raw}\n"
            f"basename searched: {base}"
        )

    def load_h5(self, path):
        with h5py.File(path, "r") as f:
            keys = list(f.keys())

            for key in ["csi", "data", "CSI", "amplitude"]:
                if key in keys:
                    return f[key][:]

            return f[keys[0]][:]

    def to_ckt(self, data):
        data = np.array(data)

        if np.iscomplexobj(data):
            data = np.abs(data)

        data = data.astype(np.float32)

        if data.ndim == 2:
            a, b = data.shape

            if a <= b:
                return data[np.newaxis, :, :]
            else:
                return data.T[np.newaxis, :, :]

        if data.ndim == 3:
            s0, s1, s2 = data.shape

            if s0 <= 8 and s1 <= self.target_subcarriers * 2:
                return data

            if s2 <= 8 and s0 <= self.target_subcarriers * 2:
                return np.transpose(data, (2, 0, 1))

            if s2 <= 8 and s1 <= self.target_subcarriers * 2:
                return np.transpose(data, (2, 1, 0))

            if s2 <= 16:
                return np.transpose(data, (2, 0, 1))

        raise ValueError(f"Unexpected CSI shape: {data.shape}")

    def standardize_subcarriers(self, x):
        C, K, T = x.shape

        if K > self.target_subcarriers:
            x = x[:, :self.target_subcarriers, :]
        elif K < self.target_subcarriers:
            pad = np.zeros((C, self.target_subcarriers - K, T), dtype=x.dtype)
            x = np.concatenate([x, pad], axis=1)

        return x

    def standardize_time(self, x):
        C, K, T = x.shape

        if T > self.target_time_len:
            x = x[:, :, :self.target_time_len]
        elif T < self.target_time_len:
            pad = np.zeros((C, K, self.target_time_len - T), dtype=x.dtype)
            x = np.concatenate([x, pad], axis=2)

        return x

    def __getitem__(self, idx):
        sample_id = str(self.sample_ids[idx])

        if sample_id not in self.meta_dict:
            raise KeyError(f"Sample ID not found in metadata: {sample_id}")

        meta = self.meta_dict[sample_id]

        file_path = self.resolve_file_path(meta["file_path"])
        csi = self.load_h5(file_path)

        x = self.to_ckt(csi)
        x = self.standardize_subcarriers(x)
        x = self.standardize_time(x)

        if self.normalize:
            x = (x - x.mean()) / (x.std() + 1e-6)

        label_name = meta["label"]

        if label_name not in self.label_mapping:
            raise KeyError(f"Label not in mapping: {label_name}")

        y = self.label_mapping[label_name]

        return torch.from_numpy(x).float(), torch.tensor(y, dtype=torch.long)

# ============================================================
# CREATE DATASETS
# ============================================================

train_dataset = CSIBenchDataset(
    ROOT,
    split="train",
    normalize=True,
    target_subcarriers=TARGET_SUBCARRIERS,
    target_time_len=TARGET_TIME_LEN,
    label_mapping=LABEL_MAPPING,
)

val_dataset = CSIBenchDataset(
    ROOT,
    split="val",
    normalize=True,
    target_subcarriers=TARGET_SUBCARRIERS,
    target_time_len=TARGET_TIME_LEN,
    label_mapping=LABEL_MAPPING,
)

test_dataset = CSIBenchDataset(
    ROOT,
    split="test",
    normalize=True,
    target_subcarriers=TARGET_SUBCARRIERS,
    target_time_len=TARGET_TIME_LEN,
    label_mapping=LABEL_MAPPING,
)

# Force one sample load to verify path resolution and shape.
sample_x, sample_y = train_dataset[0]

CSI_CHANNELS = sample_x.shape[0]

print("\n========== DATA INFO ==========")
print("Sample shape:", sample_x.shape)
print("CSI channels:", CSI_CHANNELS)
print("Num classes:", NUM_CLASSES)
print("Example label:", sample_y)

# ============================================================
# CREATE LOADERS
# ============================================================

def make_loaders(seed=42):
    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        generator=generator,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
    )

    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = make_loaders(SEED)

print("\n========== READY FOR ABLATION CELL ==========")
print("Available variables:")
print("train_loader:", type(train_loader))
print("val_loader  :", type(val_loader))
print("test_loader :", type(test_loader))
print("CSI_CHANNELS:", CSI_CHANNELS)
print("NUM_CLASSES :", NUM_CLASSES)
print("LABEL_MAPPING:", LABEL_MAPPING)

cleanup()

Using device: cuda
GPU: Tesla T4

Candidate task roots:
 - /kaggle/input/datasets/guozhenjennzhu/csi-bench/Multitask/HumanActivityRecognition

✅ Using ROOT:
/kaggle/input/datasets/guozhenjennzhu/csi-bench/Multitask/HumanActivityRecognition

========== ROOT VERIFICATION ==========
ROOT: /kaggle/input/datasets/guozhenjennzhu/csi-bench/Multitask/HumanActivityRecognition
Has metadata: True
Has splits: True
Train split: True
Val split: True
Test split: True
Metadata CSV: True
sub_Human_h5: True

========== SPLIT CHECK ==========
Train IDs: 21473
Val IDs  : 4601
Test IDs : 4602
Train-Val overlap : 0
Train-Test overlap: 0
Val-Test overlap  : 0

Label mapping:
{'jumping': 0, 'running': 1, 'seated-breathing': 2, 'walking': 3, 'wavinghand': 4}
Loaded 21473 samples for split: train
Loaded 4601 samples for split: val
Loaded 4602 samples for split: test

========== DATA INFO ==========
Sample shape: torch.Size([1, 56, 500])
CSI channels: 1
Num classes: 5
Example label: tensor(1)

========== READY F

In [2]:
# ============================================================
# Published-style Baseline Comparison for CSI-HAR Robustness
#
# Baselines:
#   B1 = Attention-BiLSTM
#   B2 = CNN-GRU-Attention
#
# Run this AFTER your pre-setup cell that creates:
#   train_loader, val_loader, test_loader
#   CSI_CHANNELS, NUM_CLASSES
#   TARGET_SUBCARRIERS, TARGET_TIME_LEN
#   LABEL_MAPPING, INV_LABEL_MAPPING
# ============================================================

import os
import gc
import json
import time
import copy
import random
import shutil
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from IPython.display import display, FileLink

# ============================================================
# CONFIG
# ============================================================

BASELINE_OUT_DIR = "/kaggle/working/drft_lstm_extra_baselines"
os.makedirs(BASELINE_OUT_DIR, exist_ok=True)

SEED = 42

BASELINE_EPOCHS = 35
BASELINE_PATIENCE = 8
MIN_DELTA = 1e-4

BATCH_SIZE = globals().get("BATCH_SIZE", 8)
EVAL_BATCH_SIZE = globals().get("EVAL_BATCH_SIZE", 32)
ACCUM_STEPS = globals().get("ACCUM_STEPS", 4)

LR = globals().get("LR", 8e-4)
WEIGHT_DECAY = globals().get("WEIGHT_DECAY", 1e-4)
MAX_GRAD_NORM = globals().get("MAX_GRAD_NORM", 1.0)
LABEL_SMOOTHING = globals().get("LABEL_SMOOTHING", 0.03)

TARGET_SUBCARRIERS = globals().get("TARGET_SUBCARRIERS", 56)
TARGET_TIME_LEN = globals().get("TARGET_TIME_LEN", 500)

DROPOUT = globals().get("DROPOUT", 0.1)

D_MODEL = 128
HIDDEN_DIM = 128
FUSION_DIM = 128

ROBUSTNESS_TRIALS = 3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RUN_BASELINES = [
    "B1_attention_bilstm",
    "B2_cnn_gru_attention",
]

required_vars = [
    "train_loader",
    "val_loader",
    "test_loader",
    "CSI_CHANNELS",
    "NUM_CLASSES",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variables from the pre-setup cell: "
        + ", ".join(missing)
        + "\nRun your CSI-Bench dataset/loading setup cell first."
    )

print("Using device:", DEVICE)
print("Running baselines:", RUN_BASELINES)
print("Output dir:", BASELINE_OUT_DIR)

# ============================================================
# REPRO / CLEANUP
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


set_seed(SEED)
cleanup()


# ============================================================
# MODEL COMPONENTS
# ============================================================

class AttentionPool1D(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.score = nn.Linear(dim, 1)

    def forward(self, x):
        # x: B x T x D
        weights = torch.softmax(self.score(x), dim=1)
        return torch.sum(weights * x, dim=1)


class AttentionBiLSTM(nn.Module):
    """
    Published-style Attention-BiLSTM / ABLSTM baseline.

    Input:
        B x C x K x T

    Processing:
        B x C x K x T
        -> B x T x (C*K)
        -> Linear projection
        -> BiLSTM
        -> Attention pooling
        -> Classifier
    """
    def __init__(
        self,
        num_classes,
        csi_channels,
        num_subcarriers,
        d_model=128,
        hidden_dim=128,
        num_layers=1,
        dropout=0.1,
    ):
        super().__init__()

        self.input_dim = csi_channels * num_subcarriers

        self.input_proj = nn.Sequential(
            nn.LayerNorm(self.input_dim),
            nn.Linear(self.input_dim, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.bilstm = nn.LSTM(
            input_size=d_model,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True,
        )

        out_dim = hidden_dim * 2

        self.attn_pool = AttentionPool1D(out_dim)

        self.head = nn.Sequential(
            nn.LayerNorm(out_dim),
            nn.Linear(out_dim, out_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(out_dim, num_classes),
        )

    def forward(self, x):
        B, C, K, T = x.shape

        # B x C x K x T -> B x T x (C*K)
        x = x.reshape(B, C * K, T).transpose(1, 2)

        z = self.input_proj(x)
        z, _ = self.bilstm(z)
        z = self.attn_pool(z)

        logits = self.head(z)
        return logits




class CNNGRUAttention(nn.Module):
    """
    Published-style CNN-GRU-Attention baseline.

    Input:
        B x C x K x T

    Processing:
        Conv2D feature extractor over subcarrier-time CSI map
        -> B x C2 x K' x T'
        -> B x T' x (C2*K')
        -> GRU
        -> Attention pooling
        -> Classifier
    """
    def __init__(
        self,
        num_classes,
        csi_channels,
        num_subcarriers,
        hidden_dim=128,
        gru_layers=1,
        dropout=0.1,
        bidirectional=False,
    ):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(csi_channels, 16, kernel_size=(3, 7), padding=(1, 3), bias=False),
            nn.BatchNorm2d(16),
            nn.GELU(),

            # Reduce time dimension, keep subcarrier resolution mostly intact.
            nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2)),

            nn.Conv2d(16, 32, kernel_size=(3, 5), padding=(1, 2), bias=False),
            nn.BatchNorm2d(32),
            nn.GELU(),

            # Reduce both dimensions lightly.
            nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2)),

            nn.Dropout2d(dropout),
        )

        # Infer CNN output dimensions dynamically.
        with torch.no_grad():
            dummy = torch.zeros(1, csi_channels, num_subcarriers, TARGET_TIME_LEN)
            out = self.cnn(dummy)
            _, c_out, k_out, t_out = out.shape

        self.cnn_channels = c_out
        self.cnn_subcarriers = k_out
        self.cnn_time = t_out
        self.gru_input_dim = c_out * k_out

        self.input_norm = nn.LayerNorm(self.gru_input_dim)

        self.gru = nn.GRU(
            input_size=self.gru_input_dim,
            hidden_size=hidden_dim,
            num_layers=gru_layers,
            batch_first=True,
            dropout=dropout if gru_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )

        out_dim = hidden_dim * (2 if bidirectional else 1)

        self.attn_pool = AttentionPool1D(out_dim)

        self.head = nn.Sequential(
            nn.LayerNorm(out_dim),
            nn.Linear(out_dim, out_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(out_dim, num_classes),
        )

    def forward(self, x):
        # x: B x C x K x T
        z = self.cnn(x)
        # z: B x C2 x K' x T'

        B, C2, K2, T2 = z.shape

        # Convert to temporal sequence:
        # B x C2 x K' x T' -> B x T' x (C2*K')
        z = z.permute(0, 3, 1, 2).contiguous()
        z = z.reshape(B, T2, C2 * K2)

        z = self.input_norm(z)
        z, _ = self.gru(z)
        z = self.attn_pool(z)

        logits = self.head(z)
        return logits


def build_baseline_model(baseline_name):
    if baseline_name == "B1_attention_bilstm":
        return AttentionBiLSTM(
            num_classes=NUM_CLASSES,
            csi_channels=CSI_CHANNELS,
            num_subcarriers=TARGET_SUBCARRIERS,
            d_model=D_MODEL,
            hidden_dim=HIDDEN_DIM,
            num_layers=1,
            dropout=DROPOUT,
        )

    if baseline_name == "B2_cnn_gru_attention":
        return CNNGRUAttention(
            num_classes=NUM_CLASSES,
            csi_channels=CSI_CHANNELS,
            num_subcarriers=TARGET_SUBCARRIERS,
            hidden_dim=HIDDEN_DIM,
            gru_layers=1,
            dropout=DROPOUT,
            bidirectional=False,
        )

    raise ValueError(f"Unknown baseline: {baseline_name}")

# ============================================================
# TRAIN / EVAL
# ============================================================

def train_one_epoch_baseline(model_obj, loader, criterion, optimizer, scaler, epoch, baseline_name):
    model_obj.train()

    running_loss = 0.0
    preds_all = []
    labels_all = []

    optimizer.zero_grad(set_to_none=True)

    progress = tqdm(loader, desc=f"{baseline_name} Epoch {epoch} Train", leave=False)

    for step, (x, y) in enumerate(progress):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=(DEVICE.type == "cuda")
        ):
            logits = model_obj(x)
            loss = criterion(logits, y)
            loss = loss / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model_obj.parameters(), MAX_GRAD_NORM)

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item() * ACCUM_STEPS

        preds = logits.argmax(dim=1)

        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(y.detach().cpu().numpy())

        progress.set_postfix(loss=f"{running_loss / (step + 1):.4f}")

        del x, y, logits, loss, preds

    return {
        "loss": running_loss / len(loader),
        "accuracy": accuracy_score(labels_all, preds_all),
        "macro_f1": f1_score(labels_all, preds_all, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels_all, preds_all, average="weighted", zero_division=0),
    }


@torch.no_grad()
def evaluate_model_loader(model_obj, loader, criterion=None, split_name="Val"):
    model_obj.eval()

    running_loss = 0.0
    preds_all = []
    labels_all = []

    progress = tqdm(loader, desc=split_name, leave=False)

    for x, y in progress:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=(DEVICE.type == "cuda")
        ):
            logits = model_obj(x)

            if criterion is not None:
                loss = criterion(logits, y)
                running_loss += loss.item()

        preds = logits.argmax(dim=1)

        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(y.detach().cpu().numpy())

        del x, y, logits, preds

    out = {
        "accuracy": accuracy_score(labels_all, preds_all),
        "macro_f1": f1_score(labels_all, preds_all, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels_all, preds_all, average="weighted", zero_division=0),
        "labels": labels_all,
        "preds": preds_all,
    }

    if criterion is not None:
        out["loss"] = running_loss / len(loader)
    else:
        out["loss"] = None

    return out


def model_size_mb(model_obj, out_dir):
    temp_path = os.path.join(out_dir, "temp_model_size.pth")
    torch.save(model_obj.state_dict(), temp_path)
    size_mb = os.path.getsize(temp_path) / (1024 ** 2)
    os.remove(temp_path)
    return size_mb


@torch.no_grad()
def profile_latency(model_obj, device, input_shape, warmup=20, runs=50):
    model_obj.eval()
    dummy = torch.randn(*input_shape).to(device)

    for _ in range(warmup):
        _ = model_obj(dummy)

    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

    start = time.perf_counter()

    for _ in range(runs):
        _ = model_obj(dummy)

    if device.type == "cuda":
        torch.cuda.synchronize()

    end = time.perf_counter()

    latency_ms = ((end - start) / runs) * 1000

    peak_mem_mb = None

    if device.type == "cuda":
        peak_mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    return latency_ms, peak_mem_mb

# ============================================================
# ROBUSTNESS PERTURBATIONS
# ============================================================

def perturb_clean(x):
    return x


def perturb_gaussian_noise(x, std=0.2):
    return x + torch.randn_like(x) * std


def perturb_random_subcarrier_mask(x, drop_prob=0.3):
    B, C, K, T = x.shape
    mask = (torch.rand(B, 1, K, 1, device=x.device) > drop_prob).float()
    return x * mask


def perturb_contiguous_subcarrier_mask(x, drop_ratio=0.3):
    B, C, K, T = x.shape
    width = max(1, int(K * drop_ratio))
    out = x.clone()

    for b in range(B):
        start = torch.randint(0, K - width + 1, (1,), device=x.device).item()
        out[b, :, start:start + width, :] = 0.0

    return out


def perturb_temporal_mask(x, drop_ratio=0.3):
    B, C, K, T = x.shape
    width = max(1, int(T * drop_ratio))
    out = x.clone()

    for b in range(B):
        start = torch.randint(0, T - width + 1, (1,), device=x.device).item()
        out[b, :, :, start:start + width] = 0.0

    return out


def perturb_crop_resize(x, keep_ratio=0.75):
    B, C, K, T = x.shape
    keep_len = max(8, int(T * keep_ratio))
    out_list = []

    for b in range(B):
        start = torch.randint(0, T - keep_len + 1, (1,), device=x.device).item()
        crop = x[b:b + 1, :, :, start:start + keep_len]
        crop = crop.reshape(1, C * K, keep_len)

        resized = F.interpolate(
            crop,
            size=T,
            mode="linear",
            align_corners=False,
        )

        resized = resized.reshape(1, C, K, T)
        out_list.append(resized)

    return torch.cat(out_list, dim=0)


def perturb_combined_harsh(x):
    x = perturb_gaussian_noise(x, std=0.20)
    x = perturb_random_subcarrier_mask(x, drop_prob=0.40)
    x = perturb_temporal_mask(x, drop_ratio=0.20)
    return x


ROBUSTNESS_CONDITIONS = [
    {"name": "clean", "fn": perturb_clean, "kwargs": {}, "trials": 1},
    {"name": "gaussian_noise_0.20", "fn": perturb_gaussian_noise, "kwargs": {"std": 0.20}, "trials": ROBUSTNESS_TRIALS},
    {"name": "random_subcarrier_mask_30", "fn": perturb_random_subcarrier_mask, "kwargs": {"drop_prob": 0.30}, "trials": ROBUSTNESS_TRIALS},
    {"name": "random_subcarrier_mask_50", "fn": perturb_random_subcarrier_mask, "kwargs": {"drop_prob": 0.50}, "trials": ROBUSTNESS_TRIALS},
    {"name": "contiguous_subcarrier_mask_30", "fn": perturb_contiguous_subcarrier_mask, "kwargs": {"drop_ratio": 0.30}, "trials": ROBUSTNESS_TRIALS},
    {"name": "temporal_mask_30", "fn": perturb_temporal_mask, "kwargs": {"drop_ratio": 0.30}, "trials": ROBUSTNESS_TRIALS},
    {"name": "crop_resize_75", "fn": perturb_crop_resize, "kwargs": {"keep_ratio": 0.75}, "trials": ROBUSTNESS_TRIALS},
    {"name": "combined_harsh", "fn": perturb_combined_harsh, "kwargs": {}, "trials": ROBUSTNESS_TRIALS},
]


@torch.no_grad()
def evaluate_under_condition(model_obj, loader, condition, trial_seed=42):
    set_seed(trial_seed)

    model_obj.eval()

    preds_all = []
    labels_all = []

    fn = condition["fn"]
    kwargs = condition["kwargs"]

    for x, y in tqdm(loader, desc=condition["name"], leave=False):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        x = fn(x, **kwargs)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=(DEVICE.type == "cuda")
        ):
            logits = model_obj(x)

        preds = logits.argmax(dim=1)

        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(y.detach().cpu().numpy())

        del x, y, logits, preds

    return {
        "accuracy": accuracy_score(labels_all, preds_all),
        "macro_f1": f1_score(labels_all, preds_all, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels_all, preds_all, average="weighted", zero_division=0),
    }


def run_robustness_for_model(model_obj, baseline_name):
    rows = []

    for condition in ROBUSTNESS_CONDITIONS:
        trial_results = []

        for t in range(condition["trials"]):
            trial_seed = SEED + 1000 * t

            metrics = evaluate_under_condition(
                model_obj=model_obj,
                loader=test_loader,
                condition=condition,
                trial_seed=trial_seed,
            )

            trial_results.append(metrics)

        accs = np.array([r["accuracy"] for r in trial_results])
        macros = np.array([r["macro_f1"] for r in trial_results])
        weighteds = np.array([r["weighted_f1"] for r in trial_results])

        rows.append({
            "baseline": baseline_name,
            "condition": condition["name"],
            "trials": condition["trials"],

            "acc_mean": float(accs.mean()),
            "acc_std": float(accs.std(ddof=1)) if len(accs) > 1 else 0.0,

            "macro_f1_mean": float(macros.mean()),
            "macro_f1_std": float(macros.std(ddof=1)) if len(macros) > 1 else 0.0,

            "weighted_f1_mean": float(weighteds.mean()),
            "weighted_f1_std": float(weighteds.std(ddof=1)) if len(weighteds) > 1 else 0.0,
        })

    robustness_df = pd.DataFrame(rows)

    clean_row = robustness_df[robustness_df["condition"] == "clean"].iloc[0]

    clean_acc = clean_row["acc_mean"]
    clean_macro = clean_row["macro_f1_mean"]
    clean_weighted = clean_row["weighted_f1_mean"]

    robustness_df["acc_drop"] = clean_acc - robustness_df["acc_mean"]
    robustness_df["macro_f1_drop"] = clean_macro - robustness_df["macro_f1_mean"]
    robustness_df["weighted_f1_drop"] = clean_weighted - robustness_df["weighted_f1_mean"]

    return robustness_df

# ============================================================
# RUN ONE BASELINE
# ============================================================

def run_single_baseline(baseline_name):
    print("\n" + "=" * 100)
    print(f"RUNNING BASELINE: {baseline_name}")
    print("=" * 100)

    set_seed(SEED)
    cleanup()

    baseline_dir = os.path.join(BASELINE_OUT_DIR, baseline_name)
    os.makedirs(baseline_dir, exist_ok=True)

    ckpt_path = os.path.join(baseline_dir, f"{baseline_name}_best.pth")
    history_path = os.path.join(baseline_dir, f"{baseline_name}_history.csv")
    robustness_path = os.path.join(baseline_dir, f"{baseline_name}_robustness.csv")
    final_json_path = os.path.join(baseline_dir, f"{baseline_name}_summary.json")
    report_path = os.path.join(baseline_dir, f"{baseline_name}_test_report.txt")
    cm_path = os.path.join(baseline_dir, f"{baseline_name}_confusion_matrix.csv")

    model_obj = build_baseline_model(baseline_name).to(DEVICE)

    params = sum(p.numel() for p in model_obj.parameters())
    print(f"Params: {params:,}")

    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    optimizer = optim.AdamW(
        model_obj.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=BASELINE_EPOCHS,
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(DEVICE.type == "cuda"),
    )

    best_val_macro_f1 = 0.0
    best_val_acc = 0.0
    best_val_weighted_f1 = 0.0
    best_epoch = 0
    patience_counter = 0
    history = []

    for epoch in range(1, BASELINE_EPOCHS + 1):
        cleanup()

        train_metrics = train_one_epoch_baseline(
            model_obj=model_obj,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            epoch=epoch,
            baseline_name=baseline_name,
        )

        val_metrics = evaluate_model_loader(
            model_obj=model_obj,
            loader=val_loader,
            criterion=criterion,
            split_name=f"{baseline_name} Epoch {epoch} Val",
        )

        scheduler.step()

        row = {
            "baseline": baseline_name,
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
        }

        history.append(row)

        print(
            f"{baseline_name} | Epoch [{epoch:02d}/{BASELINE_EPOCHS}] | "
            f"Train Acc: {row['train_acc']*100:.2f}% | "
            f"Train Macro-F1: {row['train_macro_f1']*100:.2f}% | "
            f"Val Acc: {row['val_acc']*100:.2f}% | "
            f"Val Macro-F1: {row['val_macro_f1']*100:.2f}% | "
            f"Val Weighted-F1: {row['val_weighted_f1']*100:.2f}%"
        )

        improved = row["val_macro_f1"] > best_val_macro_f1 + MIN_DELTA

        if improved:
            best_val_macro_f1 = row["val_macro_f1"]
            best_val_acc = row["val_acc"]
            best_val_weighted_f1 = row["val_weighted_f1"]
            best_epoch = epoch
            patience_counter = 0

            torch.save(
                {
                    "model_state_dict": model_obj.state_dict(),
                    "baseline": baseline_name,
                    "seed": SEED,
                    "best_epoch": best_epoch,
                    "best_val_acc": best_val_acc,
                    "best_val_macro_f1": best_val_macro_f1,
                    "best_val_weighted_f1": best_val_weighted_f1,
                    "config": {
                        "target_subcarriers": TARGET_SUBCARRIERS,
                        "target_time_len": TARGET_TIME_LEN,
                        "csi_channels": CSI_CHANNELS,
                        "num_classes": NUM_CLASSES,
                        "d_model": D_MODEL,
                        "hidden_dim": HIDDEN_DIM,
                        "dropout": DROPOUT,
                        "baseline": baseline_name,
                    },
                    "history": history,
                },
                ckpt_path,
            )

            print(
                f"Saved best {baseline_name} | "
                f"Epoch {best_epoch} | "
                f"Val Acc {best_val_acc*100:.2f}% | "
                f"Val Macro-F1 {best_val_macro_f1*100:.2f}%"
            )

        else:
            patience_counter += 1
            print(f"No improvement. Patience: {patience_counter}/{BASELINE_PATIENCE}")

        if patience_counter >= BASELINE_PATIENCE:
            print(f"Early stopping {baseline_name} at epoch {epoch}. Best epoch: {best_epoch}")
            break

    pd.DataFrame(history).to_csv(history_path, index=False)

    # Load best checkpoint.
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model_obj.load_state_dict(ckpt["model_state_dict"])
    model_obj.to(DEVICE)
    model_obj.eval()

    val_final = evaluate_model_loader(
        model_obj=model_obj,
        loader=val_loader,
        criterion=criterion,
        split_name=f"{baseline_name} Final Val",
    )

    test_final = evaluate_model_loader(
        model_obj=model_obj,
        loader=test_loader,
        criterion=criterion,
        split_name=f"{baseline_name} Final Test",
    )

    # Reports.
    if "INV_LABEL_MAPPING" in globals():
        target_names = [INV_LABEL_MAPPING[i] for i in range(NUM_CLASSES)]
    else:
        target_names = [str(i) for i in range(NUM_CLASSES)]

    test_report = classification_report(
        test_final["labels"],
        test_final["preds"],
        target_names=target_names,
        digits=4,
        zero_division=0,
    )

    test_cm = confusion_matrix(
        test_final["labels"],
        test_final["preds"],
    )

    with open(report_path, "w") as f:
        f.write(test_report)

    pd.DataFrame(test_cm, index=target_names, columns=target_names).to_csv(cm_path)

    # Edge profile.
    edge_input_shape = (1, CSI_CHANNELS, TARGET_SUBCARRIERS, TARGET_TIME_LEN)

    size_mb = model_size_mb(model_obj, baseline_dir)

    cuda_latency_ms, peak_mem_mb = profile_latency(
        model_obj,
        DEVICE,
        edge_input_shape,
        warmup=20,
        runs=50,
    )

    model_cpu = copy.deepcopy(model_obj).cpu().eval()

    cpu_latency_ms, _ = profile_latency(
        model_cpu,
        torch.device("cpu"),
        edge_input_shape,
        warmup=10,
        runs=30,
    )

    # Robustness.
    print("\n" + "-" * 80)
    print(f"Robustness evaluation: {baseline_name}")
    print("-" * 80)

    robustness_df = run_robustness_for_model(model_obj, baseline_name)

    percent_robustness_df = robustness_df.copy()

    percent_cols = [
        "acc_mean", "acc_std",
        "macro_f1_mean", "macro_f1_std",
        "weighted_f1_mean", "weighted_f1_std",
        "acc_drop", "macro_f1_drop", "weighted_f1_drop",
    ]

    for c in percent_cols:
        percent_robustness_df[c] = percent_robustness_df[c] * 100

    percent_robustness_df.to_csv(robustness_path, index=False)

    # Summary.
    summary = {
        "baseline": baseline_name,
        "seed": SEED,
        "best_epoch": int(best_epoch),

        "params": int(params),
        "model_size_mb": float(size_mb),
        "cuda_latency_ms": float(cuda_latency_ms),
        "cpu_latency_ms": float(cpu_latency_ms),
        "peak_mem_mb": float(peak_mem_mb) if peak_mem_mb is not None else None,

        "val_acc": float(val_final["accuracy"]),
        "val_macro_f1": float(val_final["macro_f1"]),
        "val_weighted_f1": float(val_final["weighted_f1"]),

        "test_acc": float(test_final["accuracy"]),
        "test_macro_f1": float(test_final["macro_f1"]),
        "test_weighted_f1": float(test_final["weighted_f1"]),

        "ckpt_path": ckpt_path,
        "history_path": history_path,
        "robustness_path": robustness_path,
        "report_path": report_path,
        "confusion_matrix_path": cm_path,
    }

    with open(final_json_path, "w") as f:
        json.dump(summary, f, indent=4)

    print("\n" + "=" * 80)
    print(f"{baseline_name} FINAL SUMMARY")
    print("=" * 80)
    print(json.dumps(summary, indent=4))

    print("\nRobustness results:")
    display(percent_robustness_df)

    del model_obj, model_cpu
    cleanup()

    return summary, percent_robustness_df

# ============================================================
# RUN ALL BASELINES
# ============================================================

all_summaries = []
all_robustness = []

for baseline_name in RUN_BASELINES:
    summary, robustness_df = run_single_baseline(baseline_name)
    all_summaries.append(summary)
    all_robustness.append(robustness_df)

summary_df = pd.DataFrame(all_summaries)
robustness_all_df = pd.concat(all_robustness, ignore_index=True)

summary_path = os.path.join(
    BASELINE_OUT_DIR,
    "extra_baselines_summary.csv",
)

robustness_all_path = os.path.join(
    BASELINE_OUT_DIR,
    "extra_baselines_robustness_all.csv",
)

summary_df.to_csv(summary_path, index=False)
robustness_all_df.to_csv(robustness_all_path, index=False)

# ============================================================
# COMPACT PAPER TABLE
# ============================================================

key_conditions = [
    "clean",
    "gaussian_noise_0.20",
    "random_subcarrier_mask_30",
    "random_subcarrier_mask_50",
    "contiguous_subcarrier_mask_30",
    "temporal_mask_30",
    "crop_resize_75",
    "combined_harsh",
]

compact_rows = []

for baseline_name in RUN_BASELINES:
    srow = summary_df[summary_df["baseline"] == baseline_name].iloc[0].to_dict()
    rsub = robustness_all_df[robustness_all_df["baseline"] == baseline_name]

    row = {
        "baseline": baseline_name,
        "params": srow["params"],
        "model_size_mb": srow["model_size_mb"],
        "cuda_latency_ms": srow["cuda_latency_ms"],
        "cpu_latency_ms": srow["cpu_latency_ms"],
        "test_acc": srow["test_acc"] * 100,
        "test_macro_f1": srow["test_macro_f1"] * 100,
        "test_weighted_f1": srow["test_weighted_f1"] * 100,
    }

    for cond in key_conditions:
        cdf = rsub[rsub["condition"] == cond]

        if len(cdf) > 0:
            row[f"{cond}_weighted_f1"] = float(cdf.iloc[0]["weighted_f1_mean"])
            row[f"{cond}_drop"] = float(cdf.iloc[0]["weighted_f1_drop"])
        else:
            row[f"{cond}_weighted_f1"] = np.nan
            row[f"{cond}_drop"] = np.nan

    compact_rows.append(row)

compact_df = pd.DataFrame(compact_rows)

compact_path = os.path.join(
    BASELINE_OUT_DIR,
    "extra_baselines_compact_paper_table.csv",
)

compact_df.to_csv(compact_path, index=False)

print("\n" + "=" * 100)
print("EXTRA BASELINES SUMMARY")
print("=" * 100)
display(summary_df)

print("\n" + "=" * 100)
print("EXTRA BASELINES ROBUSTNESS")
print("=" * 100)
display(robustness_all_df)

print("\n" + "=" * 100)
print("EXTRA BASELINES COMPACT PAPER TABLE")
print("=" * 100)
display(compact_df)

# ============================================================
# ZIP RESULTS
# ============================================================

zip_path = "/kaggle/working/drft_lstm_extra_baselines_results.zip"

if os.path.exists(zip_path):
    os.remove(zip_path)

shutil.make_archive(
    base_name=zip_path.replace(".zip", ""),
    format="zip",
    root_dir=BASELINE_OUT_DIR,
)

print("\nSaved:")
print(summary_path)
print(robustness_all_path)
print(compact_path)

print("\nDownload:")
display(FileLink(zip_path))

cleanup()

Using device: cuda
Running baselines: ['B1_attention_bilstm', 'B2_cnn_gru_attention']
Output dir: /kaggle/working/drft_lstm_extra_baselines

RUNNING BASELINE: B1_attention_bilstm
Params: 339,446


B1_attention_bilstm | Epoch [01/35] | Train Acc: 60.12% | Train Macro-F1: 39.67% | Val Acc: 67.53% | Val Macro-F1: 53.12% | Val Weighted-F1: 65.02%
Saved best B1_attention_bilstm | Epoch 1 | Val Acc 67.53% | Val Macro-F1 53.12%


B1_attention_bilstm | Epoch [02/35] | Train Acc: 74.51% | Train Macro-F1: 63.96% | Val Acc: 78.57% | Val Macro-F1: 69.82% | Val Weighted-F1: 77.66%
Saved best B1_attention_bilstm | Epoch 2 | Val Acc 78.57% | Val Macro-F1 69.82%


B1_attention_bilstm | Epoch [03/35] | Train Acc: 82.27% | Train Macro-F1: 75.42% | Val Acc: 84.57% | Val Macro-F1: 77.85% | Val Weighted-F1: 83.83%
Saved best B1_attention_bilstm | Epoch 3 | Val Acc 84.57% | Val Macro-F1 77.85%


B1_attention_bilstm | Epoch [04/35] | Train Acc: 86.01% | Train Macro-F1: 80.65% | Val Acc: 85.68% | Val Macro-F1: 80.15% | Val Weighted-F1: 85.25%
Saved best B1_attention_bilstm | Epoch 4 | Val Acc 85.68% | Val Macro-F1 80.15%


...

,baseline,condition,trials,acc_mean,acc_std,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,acc_drop,macro_f1_drop,weighted_f1_drop
0,B1_attention_bilstm,clean,1,98.000869,0.000000,97.165220,0.000000,97.995363,0.000000,0.000000,0.000000,0.000000
1,B1_attention_bilstm,gaussian_noise_0.20,3,96.805737,0.199156,95.427865,0.262703,96.820256,0.194691,1.195133,1.737355,1.175107
2,B1_attention_bilstm,random_subcarrier_mask_30,3,62.509054,0.468744,52.668933,0.548356,62.525006,0.405897,35.491815,44.496288,35.470357
3,B1_attention_bilstm,random_subcarrier_mask_50,3,51.282051,0.341509,39.121316,0.122574,51.141027,0.383441,46.718818,58.043904,46.854337
4,B1_attention_bilstm,contiguous_subcarrier_mask_30,3,51.521078,0.521965,39.036473,0.359970,51.383329,0.400309,46.479791,58.128747,46.612035
5,B1_attention_bilstm,temporal_mask_30,3,92.409097,0.119678,90.983383,0.146827,92.174638,0.113557,5.591772,6.181837,5.820725
6,B1_attention_bilstm,crop_resize_75,3,96.892655,0.065189,95.754675,0.054497,96.887944,0.063104,1.108214,1.410546,1.107419
7,B1_attention_bilstm,combined_harsh,3,55.830798,0.334053,43.040979,0.755800,54.746027,0.495578,42.170071,54.124241,43.249336
8,B2_cnn_gru_attention,clean,1,96.392873,0.000000,95.020301,0.000000,96.378406,0.000000,0.000000,0.000000,0.000000
9,B2_cnn_gru_attention,gaussian_noise_0.20,3,96.045198,0.094717,94.490903,0.164442,96.039673,0.095465,0.347675,0.529398,0.338733



EXTRA BASELINES COMPACT PAPER TABLE


,baseline,params,model_size_mb,cuda_latency_ms,cpu_latency_ms,test_acc,test_macro_f1,test_weighted_f1,clean_weighted_f1,clean_drop,...,random_subcarrier_mask_50_weighted_f1,random_subcarrier_mask_50_drop,contiguous_subcarrier_mask_30_weighted_f1,contiguous_subcarrier_mask_30_drop,temporal_mask_30_weighted_f1,temporal_mask_30_drop,crop_resize_75_weighted_f1,crop_resize_75_drop,combined_harsh_weighted_f1,combined_harsh_drop
0,B1_attention_bilstm,339446,1.300617,4.326077,9.733344,98.000869,97.165220,97.995363,97.995363,0.0,...,51.141027,46.854337,51.383329,46.612035,92.174638,5.820725,96.887944,1.107419,54.746027,43.249336
1,B2_cnn_gru_attention,421430,1.616217,1.021605,20.912736,96.392873,95.020301,96.378406,96.378406,0.0,...,50.151082,46.227325,57.895771,38.482635,93.387260,2.991146,93.039548,3.338858,53.697018,42.681388



Saved:
/kaggle/working/drft_lstm_extra_baselines/extra_baselines_summary.csv
/kaggle/working/drft_lstm_extra_baselines/extra_baselines_robustness_all.csv
/kaggle/working/drft_lstm_extra_baselines/extra_baselines_compact_paper_table.csv

Download:


/kaggle/working/drft_lstm_extra_baselines_results.zip